
# Qwen3.8-27B on Kaggle 2× T4 — optimized llama.cpp API server (v4)

## v4 improvements
- Live progress for builds, downloads, model loading, tunnel startup, and tests.
- Reuse-first behavior; build/install only when needed.
- Kaggle CUDA fallback for missing `CUDA::cuda_driver`.
- Current llama.cpp `--alias` flag.
- FAST / DEEP / MAX reasoning profiles without restarting the server.
- Automatic crash diagnostics.
- Named Cloudflare Tunnel preferred; Quick Tunnel fallback for testing.
- Harness-ready OpenAI-compatible configuration.


In [1]:

# ============================================================
# 1. USER CONFIG
# ============================================================
MODEL_REPO = "unsloth/Qwen3.8-27B-GGUF"
MODEL_FILE = "Qwen3.8-27B-UD-Q4_K_M.gguf"
MODEL_ALIAS = "qwen3.8-27b"

CTX_SIZE = 65536
PARALLEL = 1
BATCH_SIZE = 2048
UBATCH_SIZE = 512
KV_CACHE_TYPE_K = "q8_0"
KV_CACHE_TYPE_V = "q8_0"
FLASH_ATTN = "on"
TENSOR_SPLIT = "1,1"
SPLIT_MODE = "layer"
GPU_LAYERS = "all"
CACHE_REUSE_MIN_TOKENS = 256

REASONING_MODE = "auto"
REASONING_FORMAT = "deepseek"

HOST = "127.0.0.1"
PORT = 8080

# auto = named if token exists, otherwise Quick Tunnel
TUNNEL_MODE = "auto"

ENABLE_MTP = False
MTP_REPO = "unsloth/Qwen3.8-27B-GGUF"
MTP_FILE = "Qwen3.8-27B-MTP-Q4_0.gguf"

WORK = "/kaggle/working/qwen38_server"
MODEL_DIR = f"{WORK}/models"
LOG_DIR = f"{WORK}/logs"

print("[CONFIG] model:", MODEL_FILE)
print("[CONFIG] context:", CTX_SIZE)
print("[CONFIG] tunnel:", TUNNEL_MODE)


[CONFIG] model: Qwen3.8-27B-UD-Q4_K_M.gguf
[CONFIG] context: 65536
[CONFIG] tunnel: auto


In [2]:

# ============================================================
# 2. PREFLIGHT + PROGRESS HELPERS
# ============================================================
import os, sys, json, shutil, subprocess, platform, re, secrets, time, signal, glob
from pathlib import Path
from datetime import datetime

Path(WORK).mkdir(parents=True, exist_ok=True)
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

def stamp():
    return datetime.now().strftime("%H:%M:%S")

def stage(msg):
    print(f"\n[{stamp()}] ===== {msg} =====", flush=True)

def run_live(cmd, env=None, cwd=None, check=True):
    shown = " ".join(map(str, cmd)) if isinstance(cmd, (list,tuple)) else cmd
    print(f"[{stamp()}] + {shown}", flush=True)
    t0 = time.time()
    p = subprocess.Popen(
        cmd, shell=isinstance(cmd,str),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env, cwd=cwd
    )
    tail = []
    for line in iter(p.stdout.readline, ""):
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > 120: tail.pop(0)
    rc = p.wait()
    print(f"[{stamp()}] finished rc={rc} elapsed={time.time()-t0:.1f}s", flush=True)
    if check and rc != 0:
        raise RuntimeError("Command failed:\n" + "".join(tail[-60:]))
    return rc

def capture(cmd, env=None, timeout=60):
    return subprocess.run(
        cmd, shell=isinstance(cmd,str),
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env=env, timeout=timeout
    )

def gpu_state():
    p = capture([
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.free,utilization.gpu,power.draw",
        "--format=csv,noheader"
    ])
    print(p.stdout or p.stderr, flush=True)

stage("KAGGLE PREFLIGHT")
gpu = capture([
    "nvidia-smi",
    "--query-gpu=index,name,memory.total,memory.free,compute_cap,driver_version",
    "--format=csv,noheader"
])
print(gpu.stdout or gpu.stderr)
lines = [x for x in (gpu.stdout or "").splitlines() if x.strip()]
if len(lines) < 2:
    raise RuntimeError("Expected Kaggle T4 x2. Enable GPU T4 x2.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"



[13:03:16] ===== KAGGLE PREFLIGHT =====
0, Tesla T4, 15360 MiB, 14912 MiB, 7.5, 580.159.04
1, Tesla T4, 15360 MiB, 14912 MiB, 7.5, 580.159.04



In [3]:

# ============================================================
# 3. PYTHON DEPENDENCIES — ONLY IF MISSING
# ============================================================
stage("PYTHON DEPENDENCIES")
import importlib.util
missing = []
for mod,pkg in [("huggingface_hub","huggingface_hub"),("hf_xet","hf_xet"),("requests","requests")]:
    if importlib.util.find_spec(mod) is None:
        missing.append(pkg)
if missing:
    run_live([sys.executable,"-m","pip","install","-U",*missing])
else:
    print("[SKIP] dependencies already installed")
import requests
from huggingface_hub import hf_hub_download



[13:03:16] ===== PYTHON DEPENDENCIES =====
[SKIP] dependencies already installed


In [4]:

# ============================================================
# 4. SECRETS — Kaggle secrets first; safe fallbacks second
# ============================================================

def kaggle_secret(name, default=None):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        return value if value else default
    except Exception:
        return default

HF_TOKEN = kaggle_secret("HF_TOKEN")
LLAMA_API_KEY = kaggle_secret("LLAMA_API_KEY")
CF_TUNNEL_TOKEN = kaggle_secret("CLOUDFLARE_TUNNEL_TOKEN")
PUBLIC_BASE_URL = kaggle_secret("PUBLIC_BASE_URL")

if not LLAMA_API_KEY:
    LLAMA_API_KEY = "kaggle-" + secrets.token_urlsafe(32)
    print("No LLAMA_API_KEY secret found; generated an ephemeral key for this session.")

print("HF token:", "configured" if HF_TOKEN else "not needed / not configured")
print("Llama API key:", "configured")
print("Cloudflare named tunnel token:", "configured" if CF_TUNNEL_TOKEN else "not configured")
print("Public base URL:", PUBLIC_BASE_URL or "not configured")

# Do not print the actual key here. A later cell prints the final client config
# only when you explicitly run it.


No LLAMA_API_KEY secret found; generated an ephemeral key for this session.
HF token: not needed / not configured
Llama API key: configured
Cloudflare named tunnel token: not configured
Public base URL: not configured


In [5]:

# ============================================================
# 5. LLAMA.CPP — REUSE -> PREBUILT -> BUILD FALLBACK
# ============================================================
stage("LLAMA.CPP SETUP")

def locate_llama_server():
    candidates = [
        shutil.which("llama-server"),
        str(Path.home()/".local/bin/llama-server"),
        f"{WORK}/llama.cpp/build/bin/llama-server",
    ] + glob.glob("/kaggle/input/**/llama-server", recursive=True)
    for p in candidates:
        if p and Path(p).is_file():
            try: os.chmod(p,0o755)
            except: pass
            return p
    return None

def binary_has_cuda(binary):
    if not binary or not Path(binary).is_file(): return False
    p = capture([binary,"--list-devices"], timeout=45)
    print((p.stdout or "")[-4000:])
    u = (p.stdout or "").upper()
    return "CUDA" in u and ("NVIDIA" in u or "CUDA0" in u or "GPU0" in u)

LLAMA_SERVER = locate_llama_server()
if LLAMA_SERVER and binary_has_cuda(LLAMA_SERVER):
    print("[REUSE]", LLAMA_SERVER)
else:
    stage("TRY OFFICIAL PREBUILT")
    run_live("curl -LsSf https://llama.app/install.sh | sh", check=False)
    LLAMA_SERVER = locate_llama_server()

if not (LLAMA_SERVER and binary_has_cuda(LLAMA_SERVER)):
    stage("CUDA BUILD REQUIRED")
    nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if Path("/usr/local/cuda/bin/nvcc").exists() else None)
    if not nvcc: raise RuntimeError("nvcc not found")
    cuda_root = str(Path(nvcc).resolve().parent.parent)

    required = {"cmake":"cmake","ninja":"ninja-build","git":"git","pkg-config":"pkg-config"}
    pkgs = [pkg for binary,pkg in required.items() if shutil.which(binary) is None]
    if not Path("/usr/include/openssl/ssl.h").exists(): pkgs.append("libssl-dev")
    if pkgs:
        stage("INSTALL MISSING BUILD TOOLS")
        run_live("apt-get update -qq")
        run_live(["apt-get","install","-y","--no-install-recommends",*pkgs])
    else:
        print("[SKIP] build tools already present")

    src = Path(WORK)/"llama.cpp"
    build = src/"build"
    if not (src/".git").exists():
        stage("CLONE LLAMA.CPP")
        if src.exists(): shutil.rmtree(src)
        run_live(["git","clone","--depth","1","https://github.com/ggml-org/llama.cpp.git",str(src)])
    else:
        print("[REUSE] source:", src)

    if build.exists(): shutil.rmtree(build)
    build.mkdir(parents=True, exist_ok=True)

    cmake_cmd = [
        "cmake","-S",str(src),"-B",str(build),"-G","Ninja",
        "-DGGML_CUDA=ON","-DGGML_NATIVE=OFF",
        "-DCMAKE_CUDA_ARCHITECTURES=75",
        f"-DCMAKE_CUDA_COMPILER={nvcc}",
        "-DCMAKE_BUILD_TYPE=Release",
        "-DLLAMA_BUILD_TESTS=OFF","-DLLAMA_BUILD_EXAMPLES=OFF",
        "-DLLAMA_CURL=OFF","-DGGML_CCACHE=OFF",
    ]
    env = os.environ.copy()
    env["CUDA_HOME"] = cuda_root
    env["CUDACXX"] = nvcc
    env["PATH"] = f"{Path(nvcc).parent}:{env.get('PATH','')}"

    stage("CMAKE CONFIGURE")
    p = capture(cmake_cmd, env=env, timeout=180)
    print(p.stdout or "")
    if p.returncode != 0 and "CUDA::cuda_driver" in (p.stdout or "") and "target was not found" in (p.stdout or ""):
        print("[KAGGLE FIX] retrying with GGML_CUDA_NO_VMM=ON")
        shutil.rmtree(build); build.mkdir(parents=True)
        cmake_cmd += ["-DGGML_CUDA_NO_VMM=ON"]
        p = capture(cmake_cmd, env=env, timeout=180)
        print(p.stdout or "")
    if p.returncode != 0:
        raise RuntimeError("CMake configure failed")

    stage("BUILD LLAMA-SERVER — LIVE PROGRESS")
    run_live([
        "cmake","--build",str(build),
        "--target","llama-server","--config","Release","-j","4"
    ], env=env)
    LLAMA_SERVER = str(build/"bin/llama-server")

stage("FINAL VALIDATION")
if not binary_has_cuda(LLAMA_SERVER):
    raise RuntimeError("llama-server cannot see CUDA")
print("LLAMA_SERVER =", LLAMA_SERVER)



[13:03:18] ===== LLAMA.CPP SETUP =====

[13:03:18] ===== TRY OFFICIAL PREBUILT =====
[13:03:18] + curl -LsSf https://llama.app/install.sh | sh
Version: b11200
Probing CUDA...
Found: 75
Installation completed successfully

To make llama available in future sessions, add ~/.local/bin to your PATH by running:

  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bash_profile

Then open a new terminal and run:

  llama serve

To start it now without modifying your PATH, run:

  ~/.llama-app/llama serve

[13:03:36] finished rc=0 elapsed=18.4s

[13:03:36] ===== CUDA BUILD REQUIRED =====
[SKIP] build tools already present

[13:03:36] ===== CLONE LLAMA.CPP =====
[13:03:36] + git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /kaggle/working/qwen38_server/llama.cpp
Cloning into '/kaggle/working/qwen38_server/llama.cpp'...
[13:03:40] finished rc=0 elapsed=3.6s

[13:03:40] ===== CMAKE CONFIGURE =====
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is G

In [6]:

# ============================================================
# 6. MODEL — REUSE OR DOWNLOAD WITH VISIBLE PROGRESS
# ============================================================
stage("MODEL")
def find_existing_model(filename):
    for root in [MODEL_DIR,"/kaggle/input","/kaggle/working"]:
        for m in glob.glob(f"{root}/**/{filename}", recursive=True):
            try:
                if Path(m).is_file() and Path(m).stat().st_size > 1_000_000_000:
                    return m
            except: pass
    return None

MODEL_PATH = find_existing_model(MODEL_FILE)
if MODEL_PATH:
    print("[REUSE]", MODEL_PATH)
else:
    print("[DOWNLOAD]", MODEL_FILE)
    print("Hugging Face progress will appear below.")
    t0=time.time()
    MODEL_PATH = hf_hub_download(
        repo_id=MODEL_REPO, filename=MODEL_FILE,
        local_dir=MODEL_DIR, token=HF_TOKEN
    )
    print(f"[DONE] download elapsed={time.time()-t0:.1f}s")
print("MODEL_PATH =", MODEL_PATH)
print("size GiB =", round(Path(MODEL_PATH).stat().st_size/1024**3,2))



[13:29:33] ===== MODEL =====
[DOWNLOAD] Qwen3.8-27B-UD-Q4_K_M.gguf
Hugging Face progress will appear below.


Qwen3.8-27B-UD-Q4_K_M.gguf:   0%|          | 0.00/16.5G [00:00<?, ?B/s]

[DONE] download elapsed=72.9s
MODEL_PATH = /kaggle/working/qwen38_server/models/Qwen3.8-27B-UD-Q4_K_M.gguf
size GiB = 15.33



## Why these server settings?

For two equal T4s, `--split-mode layer --tensor-split 1,1` keeps the model distributed across both cards. `--gpu-layers all` attempts full GPU offload.

The notebook defaults to:

- **64K context** for agentic coding.
- **Q8 KV cache** to save substantial VRAM versus F16 while keeping a conservative cache quantization.
- **Flash Attention ON**.
- **1 parallel slot** because this server is for your own coding harness; splitting the context across multiple simultaneous slots hurts useful context and single-request performance.
- **prompt cache ON + `--cache-reuse 256`** because coding agents repeatedly resend large common prefixes.
- **reasoning `auto`** and `deepseek` formatting so compatible clients can receive `reasoning_content`.
- **MTP OFF** until its Qwen3.8 quantized correctness behavior is something you deliberately choose to test.

If 64K does not fit in a particular Kaggle session, change only `CTX_SIZE` to `49152` or `32768`; do not immediately downgrade the model quant.


In [7]:

# ============================================================
# 7. START LLAMA SERVER
# ============================================================

import signal

SERVER_PID_FILE = Path(WORK) / "llama_server.pid"
SERVER_LOG = Path(LOG_DIR) / "llama-server.log"

def kill_pidfile(path):
    path = Path(path)
    if not path.exists():
        return
    try:
        pid = int(path.read_text().strip())
        os.kill(pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(pid, 0)
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    except Exception:
        pass
    path.unlink(missing_ok=True)

kill_pidfile(SERVER_PID_FILE)

cmd = [
    LLAMA_SERVER,
    "--model", MODEL_PATH,
    "--alias", MODEL_ALIAS,
    "--host", HOST,
    "--port", str(PORT),

    "--gpu-layers", GPU_LAYERS,
    "--split-mode", SPLIT_MODE,
    "--tensor-split", TENSOR_SPLIT,
    "--main-gpu", "0",

    "--ctx-size", str(CTX_SIZE),
    "--parallel", str(PARALLEL),
    "--batch-size", str(BATCH_SIZE),
    "--ubatch-size", str(UBATCH_SIZE),

    "--flash-attn", FLASH_ATTN,
    "--cache-type-k", KV_CACHE_TYPE_K,
    "--cache-type-v", KV_CACHE_TYPE_V,

    "--cache-prompt",
    "--cache-reuse", str(CACHE_REUSE_MIN_TOKENS),
    "--cont-batching",

    "--jinja",
    "--reasoning", REASONING_MODE,
    "--reasoning-format", REASONING_FORMAT,

    "--api-key", LLAMA_API_KEY,
    "--metrics",
    "--slots",

    "--timeout", "3600",
    "--sse-ping-interval", "15",
]

if ENABLE_MTP:
    print("WARNING: MTP is experimental here and intentionally not the default.")
    MTP_PATH = find_existing_model(MTP_FILE)
    if not MTP_PATH:
        MTP_PATH = hf_hub_download(
            repo_id=MTP_REPO,
            filename=MTP_FILE,
            local_dir=MODEL_DIR,
            token=HF_TOKEN,
        )
    cmd += [
        "--spec-draft-model", MTP_PATH,
        "--spec-type", "draft-mtp",
        "--spec-draft-n-max", "1",
        "--gpu-layers-draft", "all",
    ]

print("Launching:")
safe_cmd = ["***API_KEY***" if x == LLAMA_API_KEY else x for x in cmd]
print(" ".join(map(str, safe_cmd)))

logf = open(SERVER_LOG, "w", buffering=1)
proc = subprocess.Popen(
    cmd,
    stdout=logf,
    stderr=subprocess.STDOUT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0,1"},
)
SERVER_PID_FILE.write_text(str(proc.pid))

print("PID:", proc.pid)
print("Log:", SERVER_LOG)


Launching:
/kaggle/working/qwen38_server/llama.cpp/build/bin/llama-server --model /kaggle/working/qwen38_server/models/Qwen3.8-27B-UD-Q4_K_M.gguf --alias qwen3.8-27b --host 127.0.0.1 --port 8080 --gpu-layers all --split-mode layer --tensor-split 1,1 --main-gpu 0 --ctx-size 65536 --parallel 1 --batch-size 2048 --ubatch-size 512 --flash-attn on --cache-type-k q8_0 --cache-type-v q8_0 --cache-prompt --cache-reuse 256 --cont-batching --jinja --reasoning auto --reasoning-format deepseek --api-key ***API_KEY*** --metrics --slots --timeout 3600 --sse-ping-interval 15
PID: 4709
Log: /kaggle/working/qwen38_server/logs/llama-server.log


In [8]:

# ============================================================
# 8. LIVE MODEL-LOAD PROGRESS + HEALTH + DIAGNOSTICS
# ============================================================
stage("WAIT FOR SERVER")
LOCAL = f"http://{HOST}:{PORT}"
HEADERS = {"Authorization": f"Bearer {LLAMA_API_KEY}"}
deadline=time.time()+600
last_pos=0
started=time.time()
ready=False
last_health=None

while time.time()<deadline:
    if SERVER_LOG.exists():
        with open(SERVER_LOG,"r",errors="ignore") as f:
            f.seek(last_pos)
            new=f.read()
            last_pos=f.tell()
        if new:
            print(new,end="",flush=True)

    if proc.poll() is not None:
        print(f"\n[FAIL] server exited rc={proc.poll()}")
        break

    try:
        r=requests.get(f"{LOCAL}/health",headers=HEADERS,timeout=2)
        last_health=f"{r.status_code}: {r.text[:300]}"
        if r.status_code==200:
            ready=True
            break
    except Exception as e:
        last_health=repr(e)

    elapsed=int(time.time()-started)
    if elapsed and elapsed % 15 < 2:
        print(f"\n[{stamp()}] loading... {elapsed}s")
        gpu_state()
    time.sleep(2)

if not ready:
    print("\n=== AUTOMATIC DIAGNOSTICS ===")
    print("return code:",proc.poll())
    print("last health:",last_health)
    gpu_state()
    if SERVER_LOG.exists():
        print("\n".join(SERVER_LOG.read_text(errors="ignore").splitlines()[-120:]))
    raise RuntimeError("llama-server did not become healthy")

print(f"\n[OK] healthy after {time.time()-started:.1f}s")
gpu_state()



[13:30:46] ===== WAIT FOR SERVER =====
0.00.000.738 I srv  llama_server: initializing ...
0.00.302.026 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.329.533 I srv    load_model: loading model '/kaggle/working/qwen38_server/models/Qwen3.8-27B-UD-Q4_K_M.gguf'
0.02.311.666 W model has unused tensor blk.64.attn_norm.weight (size = 20480 bytes) -- ignoring
0.02.311.696 W model has unused tensor blk.64.post_attention_norm.weight (size = 20480 bytes) -- ignoring
0.02.311.723 W model has unused tensor blk.64.attn_q.weight (size = 51609600 bytes) -- ignoring
0.02.311.734 W model has unused tensor blk.64.attn_k.weight (size = 5570560 bytes) -- ignoring
0.02.311.747 W model has unused tensor blk.64.attn_v.weight (size = 5570560 bytes) -- ignoring
0.02.311.783 W model has unused tensor blk.64.attn_output.weight (size = 25804800 bytes) -- ignoring
0.02.311.812 W model has unused tensor blk.64.attn_q_norm.weight (size = 1024 bytes) -- ignoring
0

In [9]:

# ============================================================
# 9. LOCAL OPENAI-COMPATIBLE API TEST
# ============================================================

models = requests.get(f"{LOCAL}/v1/models", headers=HEADERS, timeout=30)
print("/v1/models:", models.status_code)
print(json.dumps(models.json(), indent=2)[:3000])

payload = {
    "model": MODEL_ALIAS,
    "messages": [
        {"role": "system", "content": "You are a concise expert software engineer."},
        {"role": "user", "content": "Reply with exactly: Kaggle llama.cpp is ready"}
    ],
    "temperature": 0,
    "max_tokens": 32,
    "stream": False,
}

t0 = time.time()
resp = requests.post(
    f"{LOCAL}/v1/chat/completions",
    headers={**HEADERS, "Content-Type": "application/json"},
    json=payload,
    timeout=180,
)
elapsed = time.time() - t0

print("HTTP:", resp.status_code)
print("Elapsed:", round(elapsed, 2), "s")
print(json.dumps(resp.json(), indent=2)[:8000])

if resp.status_code != 200:
    raise RuntimeError("Local chat-completions test failed.")


/v1/models: 200
{
  "models": [
    {
      "name": "qwen3.8-27b",
      "model": "qwen3.8-27b",
      "modified_at": "",
      "size": "",
      "digest": "",
      "type": "model",
      "description": "",
      "tags": [
        ""
      ],
      "capabilities": [
        "completion"
      ],
      "parameters": "",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "",
        "families": [
          ""
        ],
        "parameter_size": "",
        "quantization_level": ""
      }
    }
  ],
  "object": "list",
  "data": [
    {
      "id": "qwen3.8-27b",
      "aliases": [
        "qwen3.8-27b"
      ],
      "tags": [],
      "object": "model",
      "created": 1790516189,
      "owned_by": "llamacpp",
      "meta": {
        "vocab_type": 2,
        "n_vocab": 248320,
        "n_ctx": 65536,
        "n_ctx_train": 262144,
        "n_embd": 5120,
        "n_params": 27320697856,
        "size": 16453443584,
        "ftype": "Q4_K - Medi

In [10]:

# ============================================================
# 9B. REQUEST PROFILES — FAST / DEEP / MAX
# ============================================================
stage("REASONING PROFILES")
PROFILES = {
    "fast": {"reasoning_effort":"none","max_tokens":4096},
    "deep": {"reasoning_effort":"medium","max_tokens":8192},
    "max":  {"reasoning_effort":"xhigh","max_tokens":16384},
}
def chat_payload(messages, profile="fast", stream=False, **overrides):
    p={
        "model":MODEL_ALIAS,
        "messages":messages,
        "temperature":0,
        "stream":stream,
        **PROFILES[profile],
    }
    p.update(overrides)
    return p
print(json.dumps(PROFILES,indent=2))



[13:36:34] ===== REASONING PROFILES =====
{
  "fast": {
    "reasoning_effort": "none",
    "max_tokens": 4096
  },
  "deep": {
    "reasoning_effort": "medium",
    "max_tokens": 8192
  },
  "max": {
    "reasoning_effort": "xhigh",
    "max_tokens": 16384
  }
}


In [14]:
# ============================================================
# 10. INSTALL CLOUDFLARED — REUSE OR DOWNLOAD WITH PROGRESS
# ============================================================

stage("CLOUDFLARED SETUP")

CLOUDFLARED = shutil.which("cloudflared") or f"{WORK}/cloudflared"

if Path(CLOUDFLARED).exists():
    print("[REUSE]", CLOUDFLARED)
else:
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

    print("[DOWNLOAD] cloudflared")
    print("Destination:", CLOUDFLARED)

    run_live([
        "curl",
        "-L",
        "--fail",
        "--retry", "3",
        "--progress-bar",
        "-o", CLOUDFLARED,
        url,
    ])

    os.chmod(CLOUDFLARED, 0o755)

print("\n[VERIFY]")
p = capture([CLOUDFLARED, "--version"], timeout=30)
print(p.stdout or "")

if p.returncode != 0:
    raise RuntimeError("cloudflared binary validation failed")

print("[OK] cloudflared ready")


[13:38:48] ===== CLOUDFLARED SETUP =====
[DOWNLOAD] cloudflared
Destination: /kaggle/working/qwen38_server/cloudflared
[13:38:48] + curl -L --fail --retry 3 --progress-bar -o /kaggle/working/qwen38_server/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
#=#=#                                                                         

                                                                           0.0%
##                                                                         2.9%
#######                                                                   10.4%
#####################                                                     30.3%
############################################################              84.5%
######################################################################## 100.0%
[13:38:49] finished rc=0 elapsed=0.7s

[VERIFY]
cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)

[OK] cloudflared ready


In [15]:

# ============================================================
# 11. CLOUDFLARE TUNNEL — AUTO / NAMED / QUICK / NONE
# ============================================================
stage("CLOUDFLARE TUNNEL")
CF_PID_FILE=Path(WORK)/"cloudflared.pid"
CF_LOG=Path(LOG_DIR)/"cloudflared.log"
kill_pidfile(CF_PID_FILE)

effective=TUNNEL_MODE
if effective=="auto":
    effective="named" if CF_TUNNEL_TOKEN else "quick"
    print("[AUTO] using:",effective)

PUBLIC_URL=None
if effective=="none":
    print("Tunnel disabled")
elif effective=="named":
    if not CF_TUNNEL_TOKEN:
        raise RuntimeError("Missing CLOUDFLARE_TUNNEL_TOKEN")
    cf_cmd=[CLOUDFLARED,"tunnel","--no-autoupdate","run","--token",CF_TUNNEL_TOKEN]
    cf_logf=open(CF_LOG,"w",buffering=1)
    cf_proc=subprocess.Popen(cf_cmd,stdout=cf_logf,stderr=subprocess.STDOUT)
    CF_PID_FILE.write_text(str(cf_proc.pid))
    for sec in range(1,16):
        time.sleep(1)
        if cf_proc.poll() is not None: break
        if sec%3==0: print(f"connecting... {sec}s",flush=True)
    if cf_proc.poll() is not None:
        print(CF_LOG.read_text(errors="ignore")[-8000:])
        raise RuntimeError("Named tunnel exited")
    PUBLIC_URL=PUBLIC_BASE_URL.rstrip("/") if PUBLIC_BASE_URL else None
    print("[OK] named tunnel connected")
    print("PUBLIC_URL:",PUBLIC_URL or "set PUBLIC_BASE_URL secret")
elif effective=="quick":
    print("[WARNING] Quick Tunnel: testing only; SSE streaming unsupported.")
    cf_cmd=[CLOUDFLARED,"tunnel","--no-autoupdate","--url",LOCAL]
    cf_logf=open(CF_LOG,"w",buffering=1)
    cf_proc=subprocess.Popen(cf_cmd,stdout=cf_logf,stderr=subprocess.STDOUT)
    CF_PID_FILE.write_text(str(cf_proc.pid))
    pat=re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    deadline=time.time()+90
    while time.time()<deadline:
        txt=CF_LOG.read_text(errors="ignore") if CF_LOG.exists() else ""
        m=pat.search(txt)
        if m:
            PUBLIC_URL=m.group(0); break
        if cf_proc.poll() is not None: break
        print(".",end="",flush=True); time.sleep(2)
    print()
    if not PUBLIC_URL:
        print(CF_LOG.read_text(errors="ignore")[-8000:])
        raise RuntimeError("Quick Tunnel failed")
    print("[OK]",PUBLIC_URL)
else:
    raise ValueError("TUNNEL_MODE must be auto/named/quick/none")



[13:38:53] ===== CLOUDFLARE TUNNEL =====
[AUTO] using: quick
[WARNING] Quick Tunnel: testing only; SSE streaming unsupported.
...
[OK] https://stating-making-referenced-cgi.trycloudflare.com


In [16]:

# ============================================================
# 12. PUBLIC ENDPOINT TEST
# ============================================================

if not PUBLIC_URL:
    print("No known PUBLIC_URL. Set PUBLIC_BASE_URL for a named tunnel and rerun this cell.")
else:
    public_headers = {"Authorization": f"Bearer {LLAMA_API_KEY}"}

    r = requests.get(f"{PUBLIC_URL}/v1/models", headers=public_headers, timeout=30)
    print("Public /v1/models HTTP:", r.status_code)
    print(r.text[:3000])

    if r.status_code != 200:
        print("\nCloudflare log:")
        print(CF_LOG.read_text(errors="ignore")[-6000:])
        raise RuntimeError("Public API test failed.")

    print("\nPublic API is reachable.")


Public /v1/models HTTP: 200
{"models":[{"name":"qwen3.8-27b","model":"qwen3.8-27b","modified_at":"","size":"","digest":"","type":"model","description":"","tags":[""],"capabilities":["completion"],"parameters":"","details":{"parent_model":"","format":"gguf","family":"","families":[""],"parameter_size":"","quantization_level":""}}],"object":"list","data":[{"id":"qwen3.8-27b","aliases":["qwen3.8-27b"],"tags":[],"object":"model","created":1790516369,"owned_by":"llamacpp","meta":{"vocab_type":2,"n_vocab":248320,"n_ctx":65536,"n_ctx_train":262144,"n_embd":5120,"n_params":27320697856,"size":16453443584,"ftype":"Q4_K - Medium"}}]}

Public API is reachable.


In [21]:
# ============================================================
# 13. PRINT CLIENT / HARNESS / WEB UI CONFIG
# ============================================================

BASE = (PUBLIC_URL or LOCAL).rstrip("/")
OPENAI_BASE_URL = BASE + "/v1"
WEB_UI_URL = BASE

print("=" * 72)
print("QWEN3.8-27B / LLAMA.CPP")
print("=" * 72)

print("\nWEB UI")
print("URL      :", WEB_UI_URL)

print("\nOPENAI-COMPATIBLE API")
print("Base URL :", OPENAI_BASE_URL)
print("Model    :", MODEL_ALIAS)
print("API Key  :", LLAMA_API_KEY)

print("\nEnvironment variables:")
print(f'export OPENAI_BASE_URL="{OPENAI_BASE_URL}"')
print(f'export OPENAI_API_KEY="{LLAMA_API_KEY}"')
print(f'export OPENAI_MODEL="{MODEL_ALIAS}"')

print("\nGeneric provider config:")
print(json.dumps({
    "provider": "openai-compatible",
    "base_url": OPENAI_BASE_URL,
    "api_key": LLAMA_API_KEY,
    "model": MODEL_ALIAS,
}, indent=2))

print("\nUseful URLs:")
print("Web UI   :", WEB_UI_URL)
print("API      :", OPENAI_BASE_URL)
print("Models   :", OPENAI_BASE_URL + "/models")
print("Health   :", BASE + "/health")
print("Metrics  :", BASE + "/metrics")
print("Slots    :", BASE + "/slots")

print("=" * 72)

if effective == "quick":
    print("\nWARNING:")
    print("Quick Tunnel is fine for testing the Web UI and non-streaming API calls.")
    print("For harness streaming/SSE, use a named Cloudflare Tunnel.")

QWEN3.8-27B / LLAMA.CPP

WEB UI
URL      : https://stating-making-referenced-cgi.trycloudflare.com

OPENAI-COMPATIBLE API
Base URL : https://stating-making-referenced-cgi.trycloudflare.com/v1
Model    : qwen3.8-27b
API Key  : kaggle-9O0jVjRNMrvZeoWsXj6gd2OzyXiznJ6NAmb34m1EZq0

Environment variables:
export OPENAI_BASE_URL="https://stating-making-referenced-cgi.trycloudflare.com/v1"
export OPENAI_API_KEY="kaggle-9O0jVjRNMrvZeoWsXj6gd2OzyXiznJ6NAmb34m1EZq0"
export OPENAI_MODEL="qwen3.8-27b"

Generic provider config:
{
  "provider": "openai-compatible",
  "base_url": "https://stating-making-referenced-cgi.trycloudflare.com/v1",
  "api_key": "kaggle-9O0jVjRNMrvZeoWsXj6gd2OzyXiznJ6NAmb34m1EZq0",
  "model": "qwen3.8-27b"
}

Useful URLs:
Web UI   : https://stating-making-referenced-cgi.trycloudflare.com
API      : https://stating-making-referenced-cgi.trycloudflare.com/v1
Models   : https://stating-making-referenced-cgi.trycloudflare.com/v1/models
Health   : https://stating-making-referenced-


## Harness reasoning profiles

Use the same endpoint and choose per request:

- `reasoning_effort: "none"` — routine coding/editing
- `reasoning_effort: "medium"` — debugging/architecture
- `reasoning_effort: "xhigh"` — rare maximum-effort tasks

Use a named Cloudflare Tunnel for normal streaming. Quick Tunnel is only for non-streaming testing.


In [18]:

# ============================================================
# 14. OPTIONAL STREAMING TEST
# ============================================================
# Use this with a named tunnel or locally. Quick Tunnel does not support SSE.

TEST_BASE = PUBLIC_URL if (PUBLIC_URL and TUNNEL_MODE == "named") else LOCAL

stream_payload = {
    "model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "Count from 1 to 10, one number at a time."}],
    "temperature": 0,
    "max_tokens": 64,
    "stream": True,
}

with requests.post(
    f"{TEST_BASE}/v1/chat/completions",
    headers={**HEADERS, "Content-Type": "application/json"},
    json=stream_payload,
    stream=True,
    timeout=180,
) as r:
    print("HTTP:", r.status_code)
    r.raise_for_status()
    seen = 0
    for raw in r.iter_lines(decode_unicode=True):
        if raw:
            print(raw[:500])
            seen += 1
        if seen >= 12:
            break


HTTP: 200
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"role":"assistant","content":null}}],"created":1790516380,"id":"chatcmpl-dc6SuxMTN2FJu1dW3HWc7xOtotj5c2r9","model":"qwen3.8-27b","system_fingerprint":"b1-9adc7f4","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"reasoning_content":"We"}}],"created":1790516380,"id":"chatcmpl-dc6SuxMTN2FJu1dW3HWc7xOtotj5c2r9","model":"qwen3.8-27b","system_fingerprint":"b1-9adc7f4","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"reasoning_content":" need"}}],"created":1790516380,"id":"chatcmpl-dc6SuxMTN2FJu1dW3HWc7xOtotj5c2r9","model":"qwen3.8-27b","system_fingerprint":"b1-9adc7f4","object":"chat.completion.chunk"}
data: {"choices":[{"finish_reason":null,"index":0,"delta":{"reasoning_content":" to"}}],"created":1790516380,"id":"chatcmpl-dc6SuxMTN2FJu1dW3HWc7xOtotj5c2r9","model":"qwen3.8-27b","system_fingerprint":"b1-9adc7f4","object":"chat.completi

In [19]:

# ============================================================
# 15. FAST CODING BENCHMARK
# ============================================================
stage("FAST CODING BENCHMARK")
bench_payload=chat_payload([{
    "role":"user",
    "content":"Write a NestJS Prisma service method that returns the latest 20 events ordered by createdAt descending. Return only TypeScript code."
}],profile="fast",stream=False,max_tokens=1024)

t0=time.time()
r=requests.post(
    f"{LOCAL}/v1/chat/completions",
    headers={**HEADERS,"Content-Type":"application/json"},
    json=bench_payload,timeout=300
)
print("HTTP:",r.status_code)
print("Wall time:",round(time.time()-t0,2),"s")
data=r.json()
print(json.dumps(data,indent=2)[:14000])
tim=data.get("timings",{})
if tim:
    print("Prompt tok/s:",round(tim.get("prompt_per_second",0),2))
    print("Generation tok/s:",round(tim.get("predicted_per_second",0),2))
gpu_state()



[13:39:44] ===== FAST CODING BENCHMARK =====
HTTP: 200
Wall time: 8.66 s
{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "```typescript\nimport { Injectable } from '@nestjs/common';\nimport { PrismaService } from './prisma.service';\n\n@Injectable()\nexport class EventService {\n  constructor(private readonly prisma: PrismaService) {}\n\n  async getLatestEvents(): Promise<any[]> {\n    return this.prisma.event.findMany({\n      orderBy: {\n        createdAt: 'desc',\n      },\n      take: 20,\n    });\n  }\n}\n```"
      }
    }
  ],
  "created": 1790516392,
  "model": "qwen3.8-27b",
  "system_fingerprint": "b1-9adc7f4",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 103,
    "prompt_tokens": 38,
    "total_tokens": 141,
    "prompt_tokens_details": {
      "cached_tokens": 0
    }
  },
  "id": "chatcmpl-pSLi3t4gfqpZjtVJAcJ4XUAUR8myCb22",
  "timings": {
    "cache_n": 0,
  

In [20]:

# ============================================================
# 16. DIAGNOSTICS / LOG TAIL
# ============================================================

print("=== llama-server ===")
if SERVER_LOG.exists():
    print("\n".join(SERVER_LOG.read_text(errors="ignore").splitlines()[-120:]))

print("\n=== cloudflared ===")
if CF_LOG.exists():
    print("\n".join(CF_LOG.read_text(errors="ignore").splitlines()[-80:]))

print("\n=== GPUs ===")
run(["nvidia-smi"], check=False)


=== llama-server ===
0.00.000.738 I srv  llama_server: initializing ...
0.00.302.026 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.329.533 I srv    load_model: loading model '/kaggle/working/qwen38_server/models/Qwen3.8-27B-UD-Q4_K_M.gguf'
0.02.311.666 W model has unused tensor blk.64.attn_norm.weight (size = 20480 bytes) -- ignoring
0.02.311.696 W model has unused tensor blk.64.post_attention_norm.weight (size = 20480 bytes) -- ignoring
0.02.311.723 W model has unused tensor blk.64.attn_q.weight (size = 51609600 bytes) -- ignoring
0.02.311.734 W model has unused tensor blk.64.attn_k.weight (size = 5570560 bytes) -- ignoring
0.02.311.747 W model has unused tensor blk.64.attn_v.weight (size = 5570560 bytes) -- ignoring
0.02.311.783 W model has unused tensor blk.64.attn_output.weight (size = 25804800 bytes) -- ignoring
0.02.311.812 W model has unused tensor blk.64.attn_q_norm.weight (size = 1024 bytes) -- ignoring
0.02.311.825 W model

NameError: name 'run' is not defined

In [ ]:

# ============================================================
# 17. STOP SERVICES
# ============================================================
# Run this when you want a clean shutdown without restarting the Kaggle kernel.

kill_pidfile(CF_PID_FILE)
kill_pidfile(SERVER_PID_FILE)
print("Stopped cloudflared and llama-server.")



## Recommended operating profile

For your coding harness, start with the notebook exactly as configured:

- `UD-Q4_K_M`
- `CTX_SIZE=65536`
- `PARALLEL=1`
- Q8 KV cache
- reasoning `auto`
- named Cloudflare Tunnel
- prompt caching/reuse enabled

### If VRAM is tight

Reduce in this order:

1. `CTX_SIZE = 49152`
2. `CTX_SIZE = 32768`
3. Change KV cache to `q4_0`

Do **not** immediately switch the 27B model down to Q3 just to preserve an oversized context you do not actually use.

### If you want more throughput for two simultaneous callers

Use `PARALLEL=2`, but remember that concurrency is not free: it increases cache pressure and reduces the useful per-request context budget. For one local development harness, `1` is the better default.

### About MTP

The switch exists for experimentation, but remains disabled by default. Reliability matters more than a synthetic token/s win when this endpoint is being trusted to edit code.

### Sources used to design this notebook

- llama.cpp multi-GPU documentation: `https://github.com/ggml-org/llama.cpp/blob/master/docs/multi-gpu.md`
- llama.cpp server documentation: `https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md`
- llama.cpp speculative decoding documentation: `https://github.com/ggml-org/llama.cpp/blob/master/docs/speculative.md`
- Unsloth Qwen3.8-27B GGUF repository: `https://huggingface.co/unsloth/Qwen3.8-27B-GGUF`
- Cloudflare Tunnel docs: `https://developers.cloudflare.com/tunnel/`
